This notebook combines the downloaded Sentinel-1 and Sentinel-2 global mosaic data by tile to create uniform multiband rasters for model building and inference. 

In [ ]:
import xarray as xr
import rioxarray
from pathlib import Path
from dask.distributed import Client, LocalCluster
import dask
import numpy as np
import rasterio

In [ ]:
# create Dask cluster and client
cluster = LocalCluster(processes=True, n_workers=8, memory_limit='32GB')
client = Client(cluster)
print(cluster.dashboard_link)

In [ ]:
# set variables
years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

S2_BANDS = ["b02", "b03", "b04", "b08"]
S2_N_QUARTERS = 4

S1_POLS = ["vv", "vh"]
S1_MONTHS = ["4", "5", "6", "7", "8", "9"]

CHUNK_SIZE = {"x": 1024, "y": 1024}

In [ ]:
for year in years:
    S2_DIR = Path(f"../data/grasslvnd/s2_{year}")
    S1_DIR = Path(f"../data/grasslvnd/s1_{year}")
    OUT_DIR = Path(f"../data/grasslvnd/combined_{year}")

    OUT_DIR.mkdir(exist_ok=True, parents=True)

    # function to load Sentinel-2 tile, normalize, and assign band names
    def load_s2_tile(tile_id):
        datasets = []
        for band in S2_BANDS:
            fpath = S2_DIR / f"cdse_s2_{band}_tile_{tile_id}.tif"
            da = rioxarray.open_rasterio(fpath, chunks=CHUNK_SIZE)
            da = da / 10000.0

            da = da.clip(0, 1)

            da = da.assign_coords(
                band=[f"B{band[-2:].upper()}_{i+1}" for i in range(S2_N_QUARTERS)]
            )
            datasets.append(da)

        return xr.concat(datasets, dim="band")

    # function to load Sentinel-1 tile, convert to dB, and assign band names
    def load_s1_tile(tile_id):
        datasets = []
        for pol in S1_POLS:
            fpath = S1_DIR / f"cdse_s1_{pol}_tile_{tile_id}.tif"
            da = rioxarray.open_rasterio(fpath, chunks=CHUNK_SIZE)

            da = 10 * np.log10(da.clip(min=1e-6))

            da = da.assign_coords(band=[f"{pol.upper()}_{m}" for m in S1_MONTHS])
            datasets.append(da)

        return xr.concat(datasets, dim="band")

    # function to combine Sentinel-2 and Sentinel-1 data for a tile and save as GeoTIFF
    def combine_and_save(tile_id):
        print(f"processing {tile_id}.")

        s2 = load_s2_tile(tile_id)
        s1 = load_s1_tile(tile_id)
        combined = xr.concat([s2, s1], dim="band")
        combined = combined.rio.write_crs(s2.rio.crs, inplace=True)

        band_names = [str(b) for b in combined.band.values]
        out_path = OUT_DIR / f"s2_s1_tile_{tile_id}.tif"

        height, width = combined.sizes["y"], combined.sizes["x"]
        transform = combined.rio.transform()
        crs = combined.rio.crs
        dtype = "float32"

        profile = {
            "driver": "GTiff",
            "height": height,
            "width": width,
            "count": len(band_names),
            "crs": crs,
            "transform": transform,
            "dtype": dtype,
            "compress": "deflate",
            "tiled": True,
            "BIGTIFF": "IF_SAFER",
        }

        combined_data = combined.astype("float32").compute()

        with rasterio.open(out_path, "w", **profile) as dst:
            for i, name in enumerate(band_names):
                dst.write(combined_data.isel(band=i).values, i + 1)
                dst.set_band_description(i + 1, name)

        print(f"saved {out_path}")
        return out_path

    tile_ids = sorted(
        {f.stem.split("_")[-1] for f in S2_DIR.glob("cdse_s2_b02_tile_*.tif")}
    )

    for tid in tile_ids:
        combine_and_save(tid)

In [ ]:
# close the Dask client and cluster
client.close()
cluster.close()